# Pseudobulk method benchmark

Benchmarks all matrix factorization methods against ground-truth cell-type fractions
across pseudobulk datasets (max Pearson correlation per cell type, via a greedy
one-to-one LV -> cell-type assignment), then compares CLAMPfull against its
competitors with paired statistics (Wilcoxon signed-rank tests) and a
dataset-level bootstrap on how often it ranks first.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
library(data.table)
library(ggplot2)
library(yaml)
library(here)

source(here("scripts", "pseudobulk", "common.R"))

set.seed(123)

## Settings

In [ ]:
DATASETS <- snakemake@params[["datasets"]]
METHODS  <- snakemake@params[["methods"]]
MOD_ROOT <- here(snakemake@params[["mod_root"]])
OUT_DIR  <- here(snakemake@params[["out_dir"]])

dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)
pseudobulk_dir <- function(ds) {
  cfg <- snakemake@config$datasets[[ds]]
  if (isTRUE(cfg$build_pseudobulk)) {
    file.path(MOD_ROOT, ds, 'pseudobulk')
  } else {
    here(cfg$pseudobulk_dir)
  }
}

## Score all methods across all datasets

In [ ]:
# Score each dataset-method pair by one-to-one matching LVs to true cell-type fractions,
# retaining CLAMPfull assignments and full correlation matrices.

long_rows      <- list()
assign_rows    <- list()
corr_full_rows <- list()

for (ds in DATASETS) {
  message("\n========== ", ds, " ==========")
  ds_dir <- file.path(MOD_ROOT, ds)

  p0_path <- file.path(pseudobulk_dir(ds), "truthFrac_v0.csv")
  if (!file.exists(p0_path)) {
    message("  Truth file missing for ", ds, "; skipping.")
    next
  }
  p0    <- filter_rare_cell_types(read_csv_matrix(p0_path))
  ct_v0 <- colnames(p0)
  cat("  truth: v0=", length(ct_v0), "types\n")

  for (meth in names(METHODS)) {
    b_path <- file.path(ds_dir, METHODS[[meth]])
    if (!file.exists(b_path)) {
      message("  ", meth, " B.csv not found — skipping.")
      next
    }

    fac <- tryCatch(read_B(b_path), error = function(e) {
      message("  ERROR reading ", meth, ": ", conditionMessage(e)); NULL
    })
    if (is.null(fac)) next
    cat("  ", meth, " B: ", nrow(fac), "samples x", ncol(fac), "LVs\n")

    cv0 <- score_per_ct(fac, p0, ct_v0)
    long_rows <- c(long_rows, list(data.table(
      dataset   = ds, method = meth, truth = "v0",
      cell_type = names(cv0), cor = unname(cv0)
    )))

    if (meth == "CLAMPfull") {
      res <- assign_with_margins(fac, p0, ct_v0)
      if (!is.null(res$assignment)) {
        assigns <- res$assignment
        assigns$dataset <- ds
        assign_rows <- c(assign_rows, list(as.data.table(assigns)))
      }
      if (!is.null(res$cc)) {
        cc_df <- res$cc
        cc_df$dataset <- ds
        corr_full_rows <- c(corr_full_rows, list(as.data.table(cc_df)))
      }
    }
  }
}

long           <- rbindlist(long_rows,      fill = TRUE)
lv_assignments <- rbindlist(assign_rows,    fill = TRUE)
lv_corr_full   <- rbindlist(corr_full_rows, fill = TRUE)
cat("\nLong table rows:", nrow(long), "\n")
cat("CLAMPfull LV assignments:", nrow(lv_assignments), "\n")
cat("CLAMPfull full LV x cell-type correlations:", nrow(lv_corr_full), "\n")
head(long)

## Summary table

In [ ]:
METHOD_LEVELS <- names(METHODS)


summary_dt <- long[, .(
  mean_cor   = mean(cor,   na.rm = TRUE),
  median_cor = median(cor, na.rm = TRUE),
  n_ct       = .N
), by = .(dataset, method, truth)]
setorder(summary_dt, dataset, truth, -mean_cor)

cat("\n--- per-method summary ---\n")
print(summary_dt)
cat("\nCLAMPfull assignments preview:\n")
print(head(lv_assignments, 10))

## Paired statistical analysis

- tests CLAMPfull vs. each competitor with paired Wilcoxon signed-rank tests
- adds a dataset-level bootstrap showing how often CLAMPfull ranks first

In [ ]:
cfg <- yaml::read_yaml(here("config.yaml"))
mc  <- unlist(cfg$MODEL_COLORS)
names(mc)[names(mc) == "GenomicSuperSignature"] <- "GSSig"
METHOD_COLORS <- mc[names(METHODS)]

long_box <- long[truth == "v0" & !is.na(cor)]
mean_order <- long_box[, .(m = mean(cor, na.rm = TRUE)), by = method]
setorder(mean_order, -m)
method_order <- as.character(mean_order$method)
long_box[, method := factor(method, levels = method_order)]

mean_df <- long_box[, .(mean_cor = mean(cor, na.rm = TRUE),
                        max_cor  = max(cor, na.rm = TRUE)), by = method]
mean_df[, method := factor(method, levels = method_order)]

xpos <- setNames(seq_along(method_order), method_order)
other_methods <- setdiff(method_order, "CLAMPfull")

box_theme <- theme_bw(base_size = 18) +
    theme(legend.position = "none",
          axis.text.x = element_text(angle = 35, hjust = 1, size = 14, margin = margin(t = 8)))

# q-value label formatting (plotmath, rendered via parse = TRUE downstream) for
# proper scientific notation with a real superscript. Jupyter's inline display
# renderer mis-renders the plotmath minus sign (shows as a box/tofu glyph);
# the exponent itself still renders correctly.
fmt_q <- function(q) {
  if (q >= 0.001) {
    sprintf('"%.3f"', q)
  } else {
    e_str    <- formatC(q, format = "e", digits = 1)
    parts    <- strsplit(e_str, "e")[[1]]
    mantissa <- trimws(parts[1])
    exp_val  <- as.integer(parts[2])
    sprintf('%s%%*%%10^{%d}', mantissa, exp_val)
  }
}

# Paired comparison unit: one row per (dataset, cell_type), one column per
# method. Every method is scored on the same cell types within a dataset, so
# CLAMPfull vs. a competitor is a paired sample, not two independent ones.
wide_ct <- dcast(long[truth == "v0"], dataset + cell_type ~ method, value.var = "cor")
methods_other <- setdiff(names(METHODS), "CLAMPfull")

diff_dt <- rbindlist(lapply(methods_other, function(m) {
  d <- wide_ct[!is.na(CLAMPfull) & !is.na(get(m)),
               .(dataset, cell_type, clampfull_cor = CLAMPfull, other_cor = get(m))]
  d[, method := m]
  d[, diff := clampfull_cor - other_cor]
  d
}))

# Paired Wilcoxon signed-rank effect sizes (Hodges-Lehmann estimate + 95% CI).
paired_stats <- rbindlist(lapply(methods_other, function(m) {
  d  <- diff_dt[method == m]
  wt <- wilcox.test(d$clampfull_cor, d$other_cor, paired = TRUE,
                     conf.int = TRUE, conf.level = 0.95, exact = FALSE)
  data.table(method = m, n_pairs = nrow(d),
             median_diff = median(d$diff),
             hl_estimate = unname(wt$estimate),
             ci_low = wt$conf.int[1], ci_high = wt$conf.int[2],
             p_two_sided = wt$p.value)
}))
paired_stats[, q_value := p.adjust(p_two_sided, method = "BH")]
setorder(paired_stats, -hl_estimate)

### Paired comparison

In [ ]:
wide_ct_box <- dcast(long_box, dataset + cell_type ~ method, value.var = "cor")

comp_df_paired <- rbind(
  data.table(a = "CLAMPfull", b = other_methods[1:3]),
  data.table(a = "CLAMPbase", b = "PLIER")
)
comp_df_paired[, p_raw := mapply(function(a, b) {
  d <- wide_ct_box[!is.na(get(a)) & !is.na(get(b))]
  wilcox.test(d[[a]], d[[b]], paired = TRUE, alternative = "two.sided", exact = FALSE)$p.value
}, a, b)]
comp_df_paired[, q := p.adjust(p_raw, method = "BH")]

assign_bracket_tiers <- function(comp_df, xpos) {
  comp_ord <- copy(comp_df)
  comp_ord[, x1 := xpos[a]]
  comp_ord[, x2 := xpos[b]]
  comp_ord[, left  := pmin(x1, x2)]
  comp_ord[, right := pmax(x1, x2)]
  comp_ord[, span  := right - left]
  setorder(comp_ord, span, q)

  levels_used <- list()
  comp_ord[, tier := 0L]

  for (i in seq_len(nrow(comp_ord))) {
    left  <- comp_ord$left[i]
    right <- comp_ord$right[i]
    tier  <- 1

    repeat {
      current <- if (tier <= length(levels_used)) levels_used[[tier]] else NULL
      overlaps <- FALSE

      if (!is.null(current)) {
        overlaps <- any(vapply(current, function(interval) {
          !(right < interval[1] || left > interval[2])
        }, logical(1)))
      }

      if (!overlaps) break
      tier <- tier + 1
    }

    comp_ord$tier[i] <- tier
    prior <- if (tier <= length(levels_used)) levels_used[[tier]] else list()
    levels_used[[tier]] <- c(prior, list(c(left, right)))
  }

  comp_ord
}

comp_ord <- assign_bracket_tiers(comp_df_paired, xpos)

n_tiers <- max(comp_ord$tier)
y_base  <- 1.08
y_step  <- 0.12
h       <- 0.025
y_max   <- y_base + (n_tiers - 1) * y_step + 0.14

options(repr.plot.width = 14, repr.plot.height = 7.7)

p_box_paired <- ggplot(long_box, aes(method, cor, fill = method)) +
  geom_boxplot(width = 0.5, outlier.shape = NA, color = "black",
               linewidth = 0.4, alpha = 0.8) +
  geom_jitter(width = 0.10, size = 1.3, shape = 21, fill = "white",
              color = "#333333", stroke = 0.3, alpha = 0.8) +
  geom_point(data = mean_df, aes(x = method, y = mean_cor), shape = 23,
             size = 3.2, fill = "white", color = "black", stroke = 0.7,
             inherit.aes = FALSE) +
  geom_text(data = mean_df,
            aes(x = method, y = max_cor + 0.05, label = sprintf("%.3f", mean_cor)),
            size = 5.2, fontface = "bold", inherit.aes = FALSE) +
  scale_fill_manual(values = METHOD_COLORS, na.value = "grey70") +
  scale_y_continuous(breaks = seq(0, 1, 0.25), expand = expansion(mult = c(0.02, 0.02))) +
  coord_cartesian(ylim = c(0, y_max), clip = "on") +
  labs(x = NULL, y = "Max Pearson r per cell type") +
  box_theme +
  theme(plot.margin = margin(10, 20, 5.5, 5.5))

for (i in seq_len(nrow(comp_ord))) {
  row <- comp_ord[i, ]
  y   <- y_base + (row$tier - 1) * y_step

  if (
    (row$a == "CLAMPfull" & row$b == "CLAMPbase") |
    (row$a == "CLAMPbase" & row$b == "CLAMPfull")
  ) {
    y <- y + 0.04
  }

  lbl <- fmt_q(row$q)

  p_box_paired <- p_box_paired +
    annotate("segment", x = row$left,  xend = row$left,  y = y,       yend = y + h, linewidth = 0.4) +
    annotate("segment", x = row$left,  xend = row$right, y = y + h, yend = y + h, linewidth = 0.4) +
    annotate("segment", x = row$right, xend = row$right, y = y,       yend = y + h, linewidth = 0.4) +
    annotate("text",
             x = (row$left + row$right) / 2,
             y = y + h + 0.018,
             label = lbl,
             size = 4.4,
             vjust = 0,
             parse = TRUE,
             family = "sans")
}

print(p_box_paired)

In [ ]:
## By-tissue benchmark (same comparison, faceted per tissue -- Brain and
## PBMC datasets are pooled together; confirms the pooled result in
## p_box_paired isn't driven by any single tissue)
tissue_map <- c(
  Brain_Mathys2023 = "Brain", Brain_Xiong2023 = "Brain",
  Heart_Datar2026  = "Heart",
  Lung_Sikkema2023 = "Lung",
  PBMC_1k1k        = "PBMC", PBMC_Perez2022 = "PBMC"
)
long_box[, tissue := tissue_map[dataset]]

# n = total number of samples (patients/donors) pooled into each tissue facet,
# read from each dataset's CLAMPfull B matrix row count (samples x LVs).
n_samples_ds <- vapply(DATASETS, function(ds) {
  nrow(read_B(file.path(MOD_ROOT, ds, METHODS[["CLAMPfull"]])))
}, integer(1))
n_tissue <- data.table(dataset = names(n_samples_ds), n_samples = n_samples_ds)
n_tissue[, tissue := tissue_map[dataset]]
n_tissue <- n_tissue[, .(N = sum(n_samples)), by = tissue]
tissue_labels <- setNames(sprintf("%s (n = %d)", n_tissue$tissue, n_tissue$N), n_tissue$tissue)
long_box[, tissue_label := tissue_labels[tissue]]

mean_df_tissue <- long_box[, .(mean_cor = mean(cor, na.rm = TRUE),
                                max_cor  = max(cor, na.rm = TRUE)), by = .(tissue, tissue_label, method)]
setorder(mean_df_tissue, tissue, -mean_cor)
mean_df_tissue[, tissue_method := factor(paste(tissue, method, sep = "___"),
                                          levels = paste(tissue, method, sep = "___"))]

long_box_tissue <- merge(long_box, mean_df_tissue[, .(tissue, method, tissue_method)],
                          by = c("tissue", "method"))

p_box_by_tissue <- ggplot(long_box_tissue, aes(tissue_method, cor, fill = method)) +
  geom_boxplot(width = 0.6, outlier.shape = NA, color = "black",
               linewidth = 0.3, alpha = 0.8) +
  geom_jitter(width = 0.10, size = 0.8, shape = 21, fill = "white",
              color = "#333333", stroke = 0.25, alpha = 0.7) +
  geom_point(data = mean_df_tissue, aes(x = tissue_method, y = mean_cor), shape = 23,
             size = 1.8, fill = "white", color = "black", stroke = 0.5,
             inherit.aes = FALSE) +
  geom_text(data = mean_df_tissue,
            aes(x = tissue_method, y = max_cor + 0.05, label = sprintf("%.3f", mean_cor)),
            size = 3, fontface = "bold", inherit.aes = FALSE) +
  facet_wrap(~tissue_label, scales = "free", ncol = 2) +
  coord_cartesian(ylim = c(0, NA)) +
  scale_x_discrete(labels = function(x) sub("^.*___", "", x)) +
  scale_fill_manual(values = METHOD_COLORS, na.value = "grey70") +
  labs(x = NULL, y = "Max Pearson r per cell type") +
  box_theme +
  theme(strip.text = element_text(size = 12, face = "bold"),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 10))

options(repr.plot.width = 14, repr.plot.height = 9)
print(p_box_by_tissue)

In [ ]:
cat("Paired CLAMPfull vs. top-3 competitors (Wilcoxon signed-rank, two-sided, BH-adjusted):\n")
print(comp_df_paired[, .(a, b, p_raw, q)])

cat("\nPaired CLAMPfull vs. competitor (Hodges-Lehmann effect size + 95% CI), all competitors:\n")
print(paired_stats)

### Bootstrap ranking stability

A dataset-level (cluster) bootstrap: resample the datasets with replacement,
average each method's per-cell-type cor within the resampled datasets, and record
which method comes out on top. Repeated over many resamples this gives a win rate
per method, a direct answer to "how often does CLAMPfull actually rank first?"

In [ ]:
# Cluster (dataset-level) bootstrap: cell types within a dataset aren't
# independent draws, so resample whole datasets rather than individual cell
# types, then check how often CLAMPfull has the top mean cor.
set.seed(123)
n_boot <- 2000
methods_boot <- names(METHODS)

ds_method_mean <- long[truth == "v0" & !is.na(cor), .(mean_cor = mean(cor)), by = .(dataset, method)]
ds_method_wide <- dcast(ds_method_mean, dataset ~ method, value.var = "mean_cor")
n_ds <- nrow(ds_method_wide)

win_counts <- setNames(rep(0L, length(methods_boot)), methods_boot)
for (b in seq_len(n_boot)) {
  samp <- ds_method_wide[sample.int(n_ds, size = n_ds, replace = TRUE)]
  method_means <- vapply(methods_boot, function(m) mean(samp[[m]], na.rm = TRUE), numeric(1))
  top_method <- names(which.max(method_means))
  win_counts[top_method] <- win_counts[top_method] + 1L
}
win_rate <- data.table(method = names(win_counts), win_rate = as.numeric(win_counts) / n_boot)
setorder(win_rate, -win_rate)
win_rate[, method := factor(method, levels = method)]

options(repr.plot.width = 14, repr.plot.height = 7.7)
p_winrate <- ggplot(win_rate, aes(method, win_rate, fill = method)) +
    geom_col(width = 0.6) +
    scale_fill_manual(values = METHOD_COLORS) +
    scale_y_continuous(labels = function(x) paste0(x * 100, "%"), limits = c(0, 1)) +
    labs(x = NULL, y = "Bootstrap win rate (rank 1)") +
    box_theme
print(p_winrate)

In [ ]:
cat(sprintf("Bootstrap win rate over %d dataset-level resamples:\n", n_boot))
print(win_rate)

## Save outputs

All tables and figures computed above, written together here.

In [ ]:
fwrite(long,           file.path(OUT_DIR, "benchmark_long.csv"))
fwrite(summary_dt,     file.path(OUT_DIR, "benchmark_summary.csv"))
fwrite(lv_assignments, file.path(OUT_DIR, "clampfull_lv_assignments.csv"))
fwrite(lv_corr_full[, .(dataset, LV, cell_type, cor)],
       file.path(OUT_DIR, "clampfull_lv_ct_corr_full.csv"))

fwrite(comp_df_paired[, .(a, b, p_raw, q)], file.path(OUT_DIR, "paired_top3_bracket_qvalues.csv"))
fwrite(paired_stats, file.path(OUT_DIR, "paired_wilcoxon_effect_sizes.csv"))
fwrite(win_rate,     file.path(OUT_DIR, "bootstrap_winrate.csv"))

cat("Saved all outputs to", OUT_DIR, "\n")